# 02 — Random Variables & Distributions
**References:** Casella & Berger (2002) Ch. 2–3 · DeGroot & Schervish (2012) Ch. 3–4

## Narrative thread
```
Random variable -> CDF -> PDF/PMF -> Expectation -> Moments -> MGF -> Key families
```

## 1. Random variables and CDFs

A **random variable** $X$ is a measurable function $X: \Omega \to \mathbb{R}$.

The **cumulative distribution function (CDF)** is:
$$F_X(x) = P(X \leq x), \quad x \in \mathbb{R}$$

**Properties of a valid CDF:**
1. $\lim_{x \to -\infty} F(x) = 0$ and $\lim_{x \to +\infty} F(x) = 1$
2. $F$ is non-decreasing
3. $F$ is right-continuous: $F(x) = \lim_{t \downarrow x} F(t)$

**Discrete vs continuous:**
- **Discrete:** $P(X = x_k) = p_k > 0$ for countably many $x_k$ (PMF)
- **Continuous:** $F$ is absolutely continuous — there exists $f \geq 0$ such that
$$F(x) = \int_{-\infty}^x f(t)\,dt \quad \Rightarrow \quad f(x) = F'(x) \text{ (PDF)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import gamma as gamma_func

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── CDF: discrete (Binomial) vs continuous (Normal) ──────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Binomial PMF
n_binom, p_binom = 10, 0.4
k = np.arange(0, n_binom + 1)
pmf = stats.binom.pmf(k, n_binom, p_binom)
axes[0, 0].bar(k, pmf, color='#4361ee', alpha=0.8, edgecolor='white')
axes[0, 0].set_title(f'Binomial PMF — B(n={n_binom}, p={p_binom})\n$P(X=k) = \\binom{{n}}{{k}} p^k(1-p)^{{n-k}}$')
axes[0, 0].set_xlabel('k'); axes[0, 0].set_ylabel('P(X = k)')

# Binomial CDF
cdf_binom = stats.binom.cdf(k, n_binom, p_binom)
axes[0, 1].step(k, cdf_binom, color='#f72585', lw=2.5, where='post')
axes[0, 1].set_title(f'Binomial CDF — right-continuous step function\n$F(x) = P(X \\leq x)$')
axes[0, 1].set_xlabel('x'); axes[0, 1].set_ylabel('F(x)')

# Normal PDF
x = np.linspace(-4, 4, 400)
axes[1, 0].plot(x, stats.norm.pdf(x), color='#4361ee', lw=2.5)
axes[1, 0].fill_between(x, stats.norm.pdf(x), alpha=0.15, color='#4361ee')
axes[1, 0].set_title('Standard Normal PDF $\\mathcal{N}(0,1)$\n$f(x) = \\frac{1}{\\sqrt{2\\pi}} e^{-x^2/2}$')
axes[1, 0].set_xlabel('x'); axes[1, 0].set_ylabel('f(x)')

# Normal CDF
axes[1, 1].plot(x, stats.norm.cdf(x), color='#f72585', lw=2.5)
axes[1, 1].axvline(0, color='gray', lw=1, linestyle='--', alpha=0.7)
axes[1, 1].axhline(0.5, color='gray', lw=1, linestyle='--', alpha=0.7)
axes[1, 1].set_title('Standard Normal CDF $\\Phi(x)$\n$F(x) = \\int_{-\\infty}^x f(t)\\,dt$, continuous and differentiable')
axes[1, 1].set_xlabel('x'); axes[1, 1].set_ylabel('F(x)')

plt.suptitle('Discrete vs Continuous Random Variables', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Expectation and moments

The **expectation** of $X$:
$$E[X] = \begin{cases} \sum_k x_k\, p_k & \text{discrete} \\ \int_{-\infty}^{\infty} x\, f(x)\,dx & \text{continuous} \end{cases}$$

**Linearity:** $E[aX + bY] = a\,E[X] + b\,E[Y]$ (always — independence not needed)

**$r$-th moment:** $\mu'_r = E[X^r]$

**$r$-th central moment:** $\mu_r = E[(X - \mu)^r]$

| Quantity | Formula | Notes |
|---|---|---|
| Mean | $\mu = E[X]$ | First moment |
| Variance | $\sigma^2 = E[(X-\mu)^2] = E[X^2] - \mu^2$ | Second central moment |
| Skewness | $\gamma_1 = E\!\left[\left(\frac{X-\mu}{\sigma}\right)^3\right]$ | Asymmetry |
| Kurtosis | $\gamma_2 = E\!\left[\left(\frac{X-\mu}{\sigma}\right)^4\right] - 3$ | Tail weight (excess) |

**LOTUS** (Law of the Unconscious Statistician):
$$E[g(X)] = \int_{-\infty}^{\infty} g(x)\,f(x)\,dx$$

## 3. Moment generating functions

The **MGF** of $X$ is $M_X(t) = E[e^{tX}]$, defined for $t$ in some neighborhood of 0.

**Key properties:**

$$E[X^r] = M_X^{(r)}(0) = \left.\frac{d^r}{dt^r} M_X(t)\right|_{t=0}$$

**Uniqueness theorem:** If $M_X(t) = M_Y(t)$ for all $t$ in a neighborhood of 0, then $F_X = F_Y$.

**Reproductive property:** If $X \perp Y$, then $M_{X+Y}(t) = M_X(t) \cdot M_Y(t)$.

| Distribution | MGF $M(t)$ |
|---|---|
| Bernoulli($p$) | $1-p+pe^t$ |
| Binomial($n,p$) | $(1-p+pe^t)^n$ |
| Poisson($\lambda$) | $\exp(\lambda(e^t-1))$ |
| Normal($\mu,\sigma^2$) | $\exp(\mu t + \frac{1}{2}\sigma^2 t^2)$ |
| Gamma($\alpha,\beta$) | $(1 - \beta t)^{-\alpha}$ for $t < 1/\beta$ |
| Exponential($\lambda$) | $(1 - t/\lambda)^{-1}$ for $t < \lambda$ |

In [ ]:
# ── MGF-derived moments vs scipy analytical moments ──────────────────────
from scipy.misc import derivative

# Normal(mu, sigma^2): M(t) = exp(mu*t + 0.5*sigma^2*t^2)
mu_true, sigma_true = 3.0, 2.0
mgf_normal = lambda t: np.exp(mu_true * t + 0.5 * sigma_true**2 * t**2)

E1_mgf  = derivative(mgf_normal, 0.0, dx=1e-5, n=1)
E2_mgf  = derivative(mgf_normal, 0.0, dx=1e-5, n=2)
var_mgf = E2_mgf - E1_mgf**2

dist = stats.norm(mu_true, sigma_true)

print('=== MGF derivatives vs true moments: Normal(mu=3, sigma=2) ===')
print(f"  E[X]   via MGF'(0)  = {E1_mgf:.6f}  (true = {mu_true})")
print(f"  E[X^2] via MGF''(0) = {E2_mgf:.6f}  (true = {mu_true**2 + sigma_true**2:.6f})")
print(f'  Var(X) = E[X^2]-E[X]^2 = {var_mgf:.6f}  (true = {sigma_true**2})')

# ── Key distributions: PDF shapes ────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
x_cont = np.linspace(0, 15, 400)
x_norm = np.linspace(-5, 5, 400)

configs = [
    ('Normal', x_norm,
     [stats.norm(0, 1), stats.norm(0, 2), stats.norm(2, 1)],
     ['N(0,1)', 'N(0,4)', 'N(2,1)'],
     '$f(x)=\\frac{1}{\\sigma\\sqrt{2\\pi}}e^{-\\frac{(x-\\mu)^2}{2\\sigma^2}}$'),
    ('Gamma', x_cont,
     [stats.gamma(1, scale=2), stats.gamma(2, scale=2), stats.gamma(5, scale=1)],
     ['Gamma(1,2)', 'Gamma(2,2)', 'Gamma(5,1)'],
     '$f(x)=\\frac{x^{\\alpha-1}e^{-x/\\beta}}{\\beta^\\alpha\\Gamma(\\alpha)}$'),
    ('Beta', np.linspace(0.01, 0.99, 400),
     [stats.beta(0.5, 0.5), stats.beta(2, 5), stats.beta(2, 2)],
     ['Beta(0.5,0.5)', 'Beta(2,5)', 'Beta(2,2)'],
     '$f(x)=\\frac{x^{\\alpha-1}(1-x)^{\\beta-1}}{B(\\alpha,\\beta)}$'),
    ('Chi-squared', x_cont,
     [stats.chi2(1), stats.chi2(3), stats.chi2(10)],
     ['$\\chi^2(1)$', '$\\chi^2(3)$', '$\\chi^2(10)$'],
     'Special case of Gamma: $\\chi^2(k) = \\Gamma(k/2, 2)$'),
    ('Student-t', x_norm,
     [stats.t(1), stats.t(3), stats.t(30)],
     ['t(1)', 't(3)', 't(30)≈N(0,1)'],
     '$f(x) \\propto (1+x^2/\\nu)^{-(\\nu+1)/2}$, heavier tails than Normal'),
    ('F', np.linspace(0.01, 8, 400),
     [stats.f(1, 1), stats.f(5, 5), stats.f(10, 30)],
     ['F(1,1)', 'F(5,5)', 'F(10,30)'],
     '$F(d_1,d_2) = \\chi^2(d_1)/d_1 \\div \\chi^2(d_2)/d_2$'),
]

colors = ['#4361ee', '#f72585', '#06d6a0']
for ax, (name, xv, dists, labels, subtitle) in zip(axes.flat, configs):
    for d, lbl, c in zip(dists, labels, colors):
        y = d.pdf(xv)
        y = np.clip(y, 0, 4)
        ax.plot(xv, y, lw=2, color=c, label=lbl)
    ax.set_title(f'{name}\n{subtitle}', fontsize=9)
    ax.legend(fontsize=7)
    ax.set_ylim(bottom=0)

plt.suptitle('Key probability distributions — PDFs', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Transformations of random variables

If $Y = g(X)$ and $g$ is monotone differentiable, the PDF of $Y$ is:

$$f_Y(y) = f_X(g^{-1}(y)) \left|\frac{d}{dy} g^{-1}(y)\right|$$

This is the **change-of-variables formula** — the Jacobian corrects for the stretching/compression of the transformation.

**Example:** If $X \sim \mathcal{N}(0,1)$ and $Y = e^X$ (log-normal), then:
$$f_Y(y) = \frac{1}{y\sqrt{2\pi}} e^{-(\ln y)^2/2}, \quad y > 0$$

**Probability integral transform:** If $X$ is continuous with CDF $F_X$, then $F_X(X) \sim \text{Uniform}(0,1)$. This underpins random number generation.

In [ ]:
# ── Probability integral transform verification ───────────────────────────
n = 10_000

# Draw from Gamma(3, 2) and apply its own CDF
dist_gamma = stats.gamma(a=3, scale=2)
x_sample   = dist_gamma.rvs(n)
u_sample   = dist_gamma.cdf(x_sample)  # should be Uniform(0,1)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].hist(x_sample, bins=50, color='#4361ee', alpha=0.8, density=True, edgecolor='white')
xv = np.linspace(0, 25, 300)
axes[0].plot(xv, dist_gamma.pdf(xv), 'r-', lw=2)
axes[0].set_title('Original: Gamma(3, 2)\n$X \\sim \\Gamma(\\alpha=3, \\beta=2)$')

axes[1].hist(u_sample, bins=50, color='#f72585', alpha=0.8, density=True, edgecolor='white')
axes[1].axhline(1, color='k', lw=2, linestyle='--')
axes[1].set_title('After PIT: $U = F_{\\Gamma}(X) \\sim \\text{Uniform}(0,1)$\n'
                  'CDF of any continuous RV is Uniform')

# Log-normal transformation: Y = e^X, X ~ N(0,1)
x_norm  = stats.norm.rvs(0, 1, size=n)
y_lognorm = np.exp(x_norm)
yv = np.linspace(0.01, 10, 300)
axes[2].hist(y_lognorm, bins=80, color='#06d6a0', alpha=0.8, density=True,
             edgecolor='white', range=(0, 10))
axes[2].plot(yv, stats.lognorm.pdf(yv, s=1), 'r-', lw=2, label='Theoretical')
axes[2].set_title('$Y = e^X$, $X \\sim \\mathcal{N}(0,1)$\n'
                  '$\\Rightarrow$ Log-Normal, Jacobian: $f_Y(y) = f_X(\\ln y)/y$')
axes[2].set_xlim(0, 10)
axes[2].legend()

plt.suptitle('Transformations of Random Variables', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Key inequalities

These inequalities appear constantly in proofs throughout statistical inference.

**Markov's inequality** (requires $X \geq 0$):
$$P(X \geq a) \leq \frac{E[X]}{a}$$

**Chebyshev's inequality:**
$$P(|X - \mu| \geq k\sigma) \leq \frac{1}{k^2}$$

**Jensen's inequality:** For convex $g$:
$$g(E[X]) \leq E[g(X)]$$

> **Consequence of Jensen:** Since $\log$ is concave, $E[\log X] \leq \log E[X]$ — this inequality is the foundation of the EM algorithm and of MLE optimality proofs.

| Takeaway | Formula |
|---|---|
| CDF defines the distribution completely | $F_X$ uniquely determines $P_X$ |
| MGF recovers all moments via derivatives | $E[X^r] = M^{(r)}(0)$ |
| Jacobian corrects for transformations | $f_Y(y) = f_X(g^{-1}(y))\|J\|$ |
| PIT: any continuous CDF is Uniform | Foundation of simulation |

**Next:** notebook 03 extends to multivariate distributions — joint, marginal, and conditional.